# Supplementary Figure 13 — sensitivity to EU + NA presence subsampling

Cohen's **κ** (mean ± SD across folds) vs. the fraction of EU + NA quarry presences
retained (5 % → 100 %), comparing the **EU + NA (subsampled)** regions against the
**other 13 regions**. Tests whether the dense EU/NA training data dominates global
suitability-model performance.

Plotting data: `results/figure-data/Supplementary-Figure-13.csv` — the per-(level, fold,
region) downsampling table `ds_all` produced by
`2-mining-suitability-model/6-sample-effect-test.ipynb` (this notebook only reads it & draws).

> Run in the activated `wty` kernel (this env's numpy LAPACK crashes if Python starts
> without the env activated).

In [ ]:
import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path

plt.rcParams['font.family'] = 'Arial'

In [ ]:
ROOT    = Path('..')
RESULTS = ROOT / 'results'
FIGURES = ROOT / 'figures'

# per-(level, fold, region) downsampling results saved by 6-sample-effect-test.ipynb
ds_all = pd.read_csv(RESULTS / 'figure-data' / 'Supplementary-Figure-13.csv')

EU_NA_REGIONS = [8, 14]               # Europe, Northern America
levels = sorted(ds_all['level'].unique())   # [0.05, 0.10, 0.25, 0.50, 1.00]

In [ ]:
# Per-fold mean kappa for EU+NA combined and for the other 13 regions combined
def group_mean_per_fold(df, region_set):
    sub = df[df['region'].isin(region_set)]
    return sub.groupby(['level', 'fold'])['kappa'].mean().reset_index()

eu_na_per_fold = group_mean_per_fold(ds_all, EU_NA_REGIONS)
other_per_fold = group_mean_per_fold(ds_all, [r for r in range(1, 16) if r not in EU_NA_REGIONS])

# Mean ± SD across folds at each level
def summarise(df):
    return df.groupby('level')['kappa'].agg(['mean', 'std']).reindex(levels)

eu_na_summary = summarise(eu_na_per_fold)
other_summary = summarise(other_per_fold)

fig = plt.figure(figsize=(5, 4))
ax1 = plt.subplot(111)
x = np.array(levels) * 100  # convert to percentage

ax1.errorbar(x, other_summary['mean'], yerr=other_summary['std'],
             marker='o', capsize=3, lw=1.8, color='#2E7D32',
             label='Other 13 regions')
ax1.errorbar(x, eu_na_summary['mean'], yerr=eu_na_summary['std'],
             marker='s', capsize=3, lw=1.8, color='#C62828',
             label='EU + NA (subsampled)')

ax1.set_xscale('log')
ax1.set_xticks([5, 10, 25, 50, 100])
ax1.set_xticklabels(['5%', '10%', '25%', '50%', '100%'])
ax1.set_xlabel('EU + NA presences retained')
ax1.set_ylabel("Cohen's \u03ba (mean \u00b1 SD across folds)")
ax1.axhline(0, color='gray', lw=0.5, ls='--')
ax1.legend(frameon=False, loc='best')
ax1.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES / 'Supplementary-Figure-13.jpg', dpi=600, bbox_inches='tight')
plt.show()